In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2, 3"

import sys
import numpy as np
import pandas as pd
from collections import Counter

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import savgol_filter
from sklearn.model_selection import train_test_split

import scanpy as sc
import anndata

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    HfArgumentParser,
    TrainingArguments,
    Trainer,
    default_data_collator,
)
from peft import (
    LoraConfig,
    PeftConfig,
    PeftModel,
    get_peft_model,
)

from dataclasses import dataclass, field
from typing import Optional
from enum import Enum
import evaluate

sys.path.append("..")
from lib_vqvae.vqvae import VQVAE
from lib_vqvae.dataset import *

#### Load the pre-trained cell tokenizer

In [2]:
import warnings
warnings.filterwarnings("ignore")
import argparse

parser = argparse.ArgumentParser()
parser.add_argument("--train_batch_size", type=int, default=64)
parser.add_argument("--eval_batch_size", type=int, default=2000)
parser.add_argument("--encoder_hidden_dim", type=list, default=[1600,1024,800])
parser.add_argument("--decoder_hidden_dim", type=list, default=[800,1024,1600])
parser.add_argument("--codebook_dim", type=int, default=8)
parser.add_argument("--n", type=int, default=32)
parser.add_argument("--n_codebooks", type=int, default=256)
parser.add_argument("--gamma", type=float, default=5)
parser.add_argument("--beta", type=float, default=10)
parser.add_argument('--weight_decay', type=float, default=5e-4)
parser.add_argument("--lr", type=float, default=1e-4) # 1e-3
parser.add_argument('--max_iter', type=int, default=20)
parser.add_argument('--seed', type=int, default=2024)
parser.add_argument('--gpu', type=int, default=0)
parser.add_argument("--dropout_rate", type=float, default=0.2)
parser.add_argument('--mode', type=str, default="vqvae")
parser.add_argument('--hvg', type=int, default=5000)

args = parser.parse_args([])

# -- device
if torch.cuda.is_available():
    args.device='cuda'
    torch.cuda.set_device(args.gpu)
else:
    args.device='cpu' 
print(args.device)
    
data_dim = 5000
vae = VQVAE(data_dim, args.n, args.codebook_dim, args.n_codebooks, args.encoder_hidden_dim, args.decoder_hidden_dim, args.beta, args.gamma, args.dropout_rate).to(args.device)

vae.load_state_dict(torch.load('../model/cell_tokenzier/model.pth')['model'])
vae.eval() 

cuda


VQVAE(
  (encoder): Encoder(
    (network): Sequential(
      (0): Linear(in_features=5000, out_features=1600, bias=True)
      (1): LayerNorm((1600,), eps=1e-05, elementwise_affine=True)
      (2): PReLU(num_parameters=1)
      (3): Dropout(p=0.2, inplace=False)
      (4): Linear(in_features=1600, out_features=1024, bias=True)
      (5): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (6): PReLU(num_parameters=1)
      (7): Dropout(p=0.2, inplace=False)
      (8): Linear(in_features=1024, out_features=800, bias=True)
      (9): LayerNorm((800,), eps=1e-05, elementwise_affine=True)
      (10): PReLU(num_parameters=1)
      (11): Dropout(p=0.2, inplace=False)
      (12): Linear(in_features=800, out_features=256, bias=True)
    )
  )
  (vector_quantization): VectorQuantizer(
    (embedding): Embedding(256, 8)
  )
  (decoder): Decoder(
    (network): Sequential(
      (0): Linear(in_features=256, out_features=800, bias=True)
      (1): LayerNorm((800,), eps=1e-05, elementwise

#### Prepare Model and Tokenizer
Now, we will be adding 258 new tokens as well as replace the existing pad, bos and eos tokens of the model.

In [3]:
# add new tokens
new_tokens = [f'<code_{i}>' for i in range(256)] + ['<cell_token_start>', '<cell_token_end>']

load the tokenizer and the LLM backbone

In [4]:
model_name = "your path to the LLM backbone"  # load the base LLM backbone
tokenizer_path = "your path to LLM's tokenizer"  # load the LLM toknizer

tokenizer = AutoTokenizer.from_pretrained(
    tokenizer_path,
    pad_token="<|pad|>",
    eos_token="<|endsentence|>",
    additional_special_tokens=new_tokens,
)

# Loading the model before finetune
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    low_cpu_mem_usage=True,
    device_map="auto",
)
# add new tokens to the tokenizer
model.resize_token_embeddings(len(tokenizer))

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Embedding(151925, 3584)

In [5]:
len(tokenizer)

151925

#### Load CellTok-base for finetuning

In [6]:
model = PeftModel.from_pretrained(model, "your path to CellTok-base")

In [7]:
for name, param in model.named_parameters():
    if "lora" in name and not param.requires_grad:
        print(f"LoRA Parameter {name} does not require grad. Enabling it.")
        param.requires_grad = True

LoRA Parameter base_model.model.model.embed_tokens.lora_embedding_A.default does not require grad. Enabling it.
LoRA Parameter base_model.model.model.embed_tokens.lora_embedding_B.default does not require grad. Enabling it.
LoRA Parameter base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight does not require grad. Enabling it.
LoRA Parameter base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight does not require grad. Enabling it.
LoRA Parameter base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight does not require grad. Enabling it.
LoRA Parameter base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight does not require grad. Enabling it.
LoRA Parameter base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight does not require grad. Enabling it.
LoRA Parameter base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight does not require grad. Enabling it.
LoRA Parameter base_model.model.model.layers

#### Load the dataset you want to finetune, which is processed by tutorial 2.

In [8]:
from datasets import load_dataset, load_from_disk
from tqdm import tqdm

max_length = 100
batch_size = 128

processed_datasets = load_from_disk("../data/blood/tokenized_dataset_generation/")

Loading dataset from disk:   0%|          | 0/30 [00:00<?, ?it/s]

split the dataset into training and test

In [14]:
processed_datasets.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

test_dataset = processed_datasets.shuffle(seed=42).select(range(1000))
train_dataset = processed_datasets.shuffle(seed=42).select(range(1000, min(11000, len(processed_datasets))))
eval_subset = test_dataset.select(range(100))

In [16]:
test_dataset, train_dataset

(Dataset({
     features: ['index_name', 'i', 'code', 'story', 'match', 'input_text', 'output_text', 'input_ids', 'attention_mask', 'labels'],
     num_rows: 1000
 }),
 Dataset({
     features: ['index_name', 'i', 'code', 'story', 'match', 'input_text', 'output_text', 'input_ids', 'attention_mask', 'labels'],
     num_rows: 3755
 }))

In [17]:
train_dataloader = DataLoader(
    train_dataset, shuffle=True, collate_fn=default_data_collator, batch_size=batch_size, pin_memory=True
)
next(iter(train_dataloader))

{'input_ids': tensor([[151666, 151666, 151666,  ..., 151891, 151924, 151665],
         [151666, 151666, 151666,  ..., 151866, 151924, 151665],
         [151666, 151666, 151666,  ..., 151891, 151924, 151665],
         ...,
         [151666, 151666, 151666,  ..., 151866, 151924, 151665],
         [151666, 151666, 151666,  ..., 151866, 151924, 151665],
         [151666, 151666, 151666,  ..., 151746, 151924, 151665]]),
 'attention_mask': tensor([[0, 0, 0,  ..., 1, 1, 1],
         [0, 0, 0,  ..., 1, 1, 1],
         [0, 0, 0,  ..., 1, 1, 1],
         ...,
         [0, 0, 0,  ..., 1, 1, 1],
         [0, 0, 0,  ..., 1, 1, 1],
         [0, 0, 0,  ..., 1, 1, 1]]),
 'labels': tensor([[  -100,   -100,   -100,  ..., 151891, 151924, 151665],
         [  -100,   -100,   -100,  ..., 151866, 151924, 151665],
         [  -100,   -100,   -100,  ..., 151891, 151924, 151665],
         ...,
         [  -100,   -100,   -100,  ..., 151866, 151924, 151665],
         [  -100,   -100,   -100,  ..., 151866, 15192

#### Train the model
change the output_model_path and save_model_name to your own.

In [10]:
import evaluate

eval_batch_size = 100
output_model_path = "../data/blood/" # the path to save the finetuned model
save_model_name = "finetune"  # the name for the saved model package
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

import random
random.seed(2024)
from transformers import TrainerCallback
from torch.utils.data import Subset
import wandb

def is_main_process():
    if torch.distributed.is_initialized():
        return torch.distributed.get_rank() == 0
    return True  

class CustomEvalAndLogCallback(TrainerCallback):
    def __init__(self, trainer, eval_datasets, sample_size=None, eval_step_per=100):
        self.trainer = trainer  
        self.eval_datasets = eval_datasets
        self.sample_size = sample_size
        self.eval_step_per = eval_step_per
        # self.trainer.model.to(device)

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % self.eval_step_per == 0:
            log_data = {"global_step": state.global_step}

            for ds_name, dataset_dict in self.eval_datasets.items():
                for split, dataset in dataset_dict.items():
                    if self.sample_size is not None:
                        indices = random.sample(range(len(dataset)), self.sample_size)
                        subset = Subset(dataset, indices)
                    else:
                        subset = dataset

                    eval_loss = 0.0
                    for sample in subset:
                        sample = {k: v.unsqueeze(0).to(device) for k, v in sample.items()} 
                        
                        with torch.no_grad():  
                            outputs = self.trainer.model(**sample)
                            loss = outputs.loss
                            eval_loss += loss.item()

                    avg_eval_loss = eval_loss / len(subset)
                    log_data.update({
                        f"{ds_name}/{split}/eval_loss": avg_eval_loss
                    })

            if is_main_process():  
                print(log_data)  
                wandb.log(log_data)
            print(f"Step {state.global_step}: {log_data}")

In [11]:
training_args = TrainingArguments(
    output_dir=os.path.join(output_model_path, save_model_name),
    num_train_epochs=1,
    save_total_limit=50,
    per_device_train_batch_size=batch_size,
    
    do_eval=True,
    eval_strategy="no",
    # eval_strategy="steps",
    eval_steps = 50,
    # metric_for_best_model='bleu',
    
    per_device_eval_batch_size = eval_batch_size, 
    warmup_steps=10,
    weight_decay=0.0001,
    dataloader_drop_last=True,
    bf16=True,
    logging_steps=50,
    learning_rate=1e-5,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    remove_unused_columns=False,
    push_to_hub=False,
    # hub_private_repo=True,
    save_strategy="steps",  
    save_steps=50,       
    logging_dir = './logs',
    report_to=['wandb'],
    run_name=save_model_name,
)

In [ ]:
callback = CustomEvalAndLogCallback(
    trainer=None,  
    eval_datasets={"eval": {"subset": eval_subset}},  
    sample_size=100,  
    eval_step_per=50,  
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=None, 
    callbacks=[callback],  
    data_collator=default_data_collator,
)

callback.trainer = trainer  
model.config.use_cache = False

trainer.train()

Then you can use the saved finetuned model for analysis following tutorial 3/4, according to your task.